# 이미지 생성 애플리케이션 구축 (Azure OpenAI)

LLM은 텍스트 생성에만 국한되지 않습니다. 텍스트 설명으로부터 이미지를 생성할 수도 있습니다. 이미지라는 매체는 의료기술, 건축, 관광, 게임 개발, 마케팅 등 다양한 분야에서 유용합니다. 이번 수업에서는 Microsoft Foundry에서 오늘날의 **GPT 이미지** 모델을 다룹니다.

## 학습 목표

이 수업이 끝나면 다음을 할 수 있게 됩니다:

- 이미지 생성이 무엇이며 어디에서 유용한지 설명할 수 있습니다.
- `gpt-image` 모델 계열과 기존의 DALL·E 모델과의 차이를 이해합니다.
- 이미지 생성 애플리케이션을 구축하고 이미지를 편집합니다.

## 이미지 생성이란?

이미지 생성 모델은 텍스트 프롬프트로부터 이미지를 만듭니다. `gpt-image`와 같은 최신 모델은 학습 과정에서 텍스트와 이미지 간의 관계를 배우고, 무작위 노이즈를 반복적으로 원하는 프롬프트에 맞는 이미지로 변환합니다.

잘 알려진 두 이미지 모델 계열은 다음과 같습니다:

- **`gpt-image` (OpenAI)** — 본 수업에서 사용하는 현재 세대 모델로, 텍스트에서 이미지 생성과 이미지 편집(마스크를 활용한 인페인팅)을 지원합니다.
- **Midjourney** — 자체 서비스와 Discord 기반 워크플로우를 가진 인기 있는 타사 모델입니다.

> 이전 OpenAI 이미지 모델인 — <strong>DALL·E 2</strong>와 **DALL·E 3** — 는 구형입니다. DALL·E 3는 새 배포용으로 더 이상 제공되지 않으며, `create_variation` 같은 기능은 DALL·E 2에만 있었습니다. 새 애플리케이션에는 `gpt-image` 모델을 사용하세요.

Microsoft Foundry에서는 **`gpt-image-2`** 가 최신이자 가장 강력한 이미지 모델이며 기본 권장 모델입니다. `gpt-image-1.5`와 `gpt-image-1-mini`도 일반적으로 사용 가능합니다.

> **중요:** `gpt-image` 모델은 생성된 이미지를 URL이 아닌 **base64** (`b64_json`) 형태로 반환합니다. 코드가 base64 문자열을 디코딩하여 바이트로 변환 후 저장합니다 — 다운로드 가능한 이미지 URL은 없습니다.


## 첫 번째 이미지 생성 애플리케이션 빌드하기

이미지 생성 애플리케이션을 빌드하려면 무엇이 필요할까요? 다음 라이브러리가 필요합니다:

- **python-dotenv**, 비밀 정보를 코드와 분리하여 *.env* 파일에 보관하기 위해 이 라이브러리 사용을 강력히 권장합니다.
- **openai**, OpenAI API와 상호작용할 때 사용할 라이브러리입니다.
- **pillow**, 파이썬에서 이미지를 다루기 위한 라이브러리입니다.

아직 하지 않았다면, [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) 페이지의 안내를 따라 Microsoft Foundry 리소스와 모델을 생성하세요. 모델은 <strong>gpt-image-2</strong>로 선택하세요 (최신 Azure OpenAI 이미지 모델; DALL·E는 이전 버전입니다).

1. 다음 내용을 포함하는 *.env* 파일을 만드세요:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    이 정보는 Microsoft Foundry 포털의 "배포(Deployments)" 섹션에서 해당 리소스 정보를 찾으면 됩니다.



1. 위 라이브러리들을 <em>requirements.txt</em>라는 파일에 다음과 같이 모으세요:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 다음으로, 가상 환경을 만들고 라이브러리들을 설치하세요:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windows의 경우, 다음 명령어를 사용하여 가상 환경을 생성하고 활성화하세요:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. <em>app.py</em>라는 파일에 다음 코드를 추가하세요:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # dotenv 가져오기
    dotenv.load_dotenv()

    # Azure OpenAI (Microsoft Foundry) 클라이언트 구성.
    # 이미지 모델은 최신 API 버전이 필요함 — 모델에 맞는 버전은 Foundry 문서를 확인하세요.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # 이미지 생성 API를 사용하여 이미지 생성
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 여기에 프롬프트 텍스트를 입력하세요
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # 예: "gpt-image-2"
        )
        # 저장된 이미지 디렉터리 설정
        image_dir = os.path.join(os.curdir, 'images')

        # 디렉터리가 없으면 생성
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 이미지 경로 초기화 (파일 형식은 png이어야 함)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 모델은 URL이 아닌 base64 (b64_json) 형식으로 이미지를 반환함
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 기본 이미지 뷰어에서 이미지 표시
        image = Image.open(image_path)
        image.show()

    # 예외 처리
    except BadRequestError as err:
        print(err)

    ```

이 코드를 설명하겠습니다:

- 먼저, OpenAI 라이브러리, dotenv 라이브러리, base64 모듈, Pillow 라이브러리를 포함해 필요한 라이브러리를 가져옵니다.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- 그 다음, *.env* 파일에서 환경 변수를 불러옵니다.

    ```python
    # dotenv를 가져옵니다
    dotenv.load_dotenv()
    ```

- 이후, Azure OpenAI (Microsoft Foundry) 클라이언트를 구성합니다.

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- 다음으로, 이미지를 생성합니다:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 여기에 프롬프트 텍스트를 입력하세요
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    `gpt-image` 모델은 이미지를 `data[0].b64_json` 내의 **base64** 문자열로 반환합니다. 이를 바이트로 디코딩하여 파일로 저장합니다 — 다운로드할 수 있는 URL은 없습니다.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 마지막으로, 이미지를 열고 기본 이미지 뷰어를 사용하여 표시합니다:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 이미지 생성에 대한 추가 세부사항

`images.generate`의 매개변수를 살펴보겠습니다:

- <strong>prompt</strong>는 이미지를 생성하는 데 사용되는 텍스트 프롬프트입니다. 여기서는 "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils"입니다.
- <strong>size</strong>는 생성될 이미지의 크기입니다 (예: `1024x1024`, `1536x1024`, `1024x1536`, 또는 `"auto"`).
- <strong>n</strong>은 생성할 이미지 수입니다. 여기서는 한 장을 생성합니다.
- <strong>model</strong>은 이미지 모델 배포 이름입니다 (예: `gpt-image-2`).

> 이미지 모델은 `temperature` 매개변수를 받지 않습니다 — 해당 매개변수는 텍스트 생성 제어에 사용됩니다. 다양성을 얻고 싶으면 API를 다시 호출하세요; 다양성을 줄이고 싶으면 프롬프트를 더 구체적으로 만드세요.

## 이미지 생성의 추가 기능

몇 줄의 Python 코드로 이미지를 생성하는 방법을 보았습니다. `gpt-image` 모델은 기존 이미지를 <strong>편집</strong>할 수도 있습니다. 이미지와 선택적 <strong>마스크</strong>(변경할 영역 지정), 그리고 프롬프트를 제공하면 이미지의 일부를 수정할 수 있습니다 — 예를 들어, 토끼에게 모자를 추가하는 것과 같습니다.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 편집 내용도 base64로 반환됩니다
edited_image = base64.b64decode(response.data[0].b64_json)
```

기본 이미지는 토끼만 포함하고; 최종 이미지는 토끼에게 모자가 씌워져 있습니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
